In [ ]:
# Processing monthly FSNSC files

In [1]:
import os
import xarray as xr
from utils.processing_utils import fix_months

In [2]:
# === Path Builder ===
def get_file_paths(scenario, ens_num):
    num = f"{ens_num:02d}"
    files = []

    if scenario == "ARISE":
        end = "206912" if ens_num not in [8, 9] else "207012"
        path = os.path.join(
            "/glade/campaign/cesm/collections/ARISE-SAI-1.5/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}/atm/proc/tseries/month_1/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}.cam.h0.FSNSC.203501-{end}.nc"
        )
        files.append(path)

    elif scenario == "SSP245":
        base = os.path.join(
            "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/",
            f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.FSNSC.201501-206412.nc"))
        end = "210012" if ens_num <= 5 else "206912"
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.FSNSC.206501-{end}.nc"))

    return files

In [5]:
# === Update based on your environment ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/radiation/FSNSC/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main Loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, ensemble {ens_num:02d}")
        file_list = get_file_paths(scenario, ens_num)
        datasets = []

        for f in file_list:
            if not os.path.exists(f):
                raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f)}")
            datasets.append(xr.open_dataset(f)["FSNSC"])

        # Combine files if multiple
        combined_ds = xr.concat(datasets, dim="time") if len(datasets) > 1 else datasets[0]

        if scenario == "ARISE":
            try:
                expected_end = "2070-12" if ens_num in [8, 9] else "2069-12"
                annual_fsnsc = fix_months(combined_ds, "2035-01", expected_end, scenario)
            except Exception as e:
                print(f"Error processing ensemble {ens_num:02d}: {e}")
                continue

        if scenario == "SSP245":
            try:
                expected_end = "2100-12" if ens_num <= 5 else "2069-12"
                annual_fsnsc = fix_months(combined_ds, "2015-01", expected_end, scenario)
            except Exception as e:
                print(f"Error processing ensemble {ens_num:02d}: {e}")
                continue

        # Save output
        if scenario == "ARISE":
            dates = "203501-206912"
        else:
            dates = "202001-206912"
        out_file = f"FSNSC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving monthly FSNSC to {out_path}")
        annual_fsnsc.to_netcdf(out_path)

print("Done processing all FSNSC ensembles.")

Processing ARISE, ensemble 01
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.001.cam.h0.FSNSC.203501-206912.nc
Saving monthly FSNSC to /glade/work/awells/air_quality/CESM/radiation/FSNSC/FSNSC_CESM2_ARISE_01_203501-206912.nc
Processing ARISE, ensemble 02
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.002.cam.h0.FSNSC.203501-206912.nc
Saving monthly FSNSC to /glade/work/awells/air_quality/CESM/radiation/FSNSC/FSNSC_CESM2_ARISE_02_203501-206912.nc
Processing ARISE, ensemble 03
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.003.cam.h0.FSNSC.203501-206912.nc
Saving monthly FSNSC to /glade/work/awells/air_quality/CESM/radiation/FSNSC/FSNSC_CESM2_ARISE_03_203501-206912.nc
Processing ARISE, ensemble 04
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.004.cam.h0.FSNSC.203501-206912.nc
Saving monthly FSNSC to /glade/work/awells/air_quality/CESM/radiation/FSNSC/FSNSC_CESM2_ARISE_04_203501-206912.nc
Processing ARISE, ensemble 05
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAUL